In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
import os
import numpy as np
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import torch.nn.functional as F

# ==========================================
# 0. 配置 (Configuration)
# ==========================================
DATA_DIR = '/kaggle/input/raf-db-dataset/DATASET' 
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-4
NUM_CLASSES = 7
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# [New] 对比学习参数
TEMP = 0.07         # 温度系数，越小越关注难样本
LAMBDA_CONT = 0.1   # 对比损失的权重 (0.1 是个经验值，既不喧宾夺主又能优化特征)

print(f"🔥 Running SS-VLM with Supervised Contrastive Loss on {DEVICE}...")

# ==========================================
# 1. 损失函数：Supervised Contrastive Loss
# ==========================================
class SupConLoss(nn.Module):
    """
    Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf
    让同类样本特征更近，不同类更远。
    """
    def __init__(self, temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        """
        features: [batch_size, feature_dim] (需要是 normalize 过的)
        labels: [batch_size]
        """
        device = features.device
        batch_size = features.shape[0]
        
        # 计算相似度矩阵: [B, B]
        # features 已经是 normalize 过的，dot product 就是 cosine similarity
        similarity_matrix = torch.matmul(features, features.T) / self.temperature
        
        # Mask: 同样 label 的样本为 1 (正样本对)
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        
        # Mask-out self-contrast cases: 对角线置为 0 (自己和自己不算)
        logits_mask = torch.scatter(
            torch.ones_like(mask), 
            1, 
            torch.arange(batch_size).view(-1, 1).to(device), 
            0
        )
        mask = mask * logits_mask

        # 计算 Log-Sum-Exp (分母)
        # 为了数值稳定性，减去每一行的最大值
        logits_max, _ = torch.max(similarity_matrix, dim=1, keepdim=True)
        logits = similarity_matrix - logits_max.detach()
        
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)
        
        # 计算 Mean Log-Likelihood
        # sum(mask) 可能为 0 (比如这就一个样本是该类)，加 1e-6 防止除零
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-6)
        
        loss = - mean_log_prob_pos
        loss = loss.mean()
        
        return loss

# ==========================================
# 2. 核心模块: AFRN & GeM (保持不变)
# ==========================================
class SpectralCoordinateAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        mid_channels = max(8, in_channels // reduction_ratio)
        
        # Spectral Branch
        self.avg_pool = nn.AvgPool2d(3, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, mid_channels, bias=False),
            nn.GELU(),
            nn.Linear(mid_channels, in_channels, bias=True),
            nn.Sigmoid()
        )
        
        # Coordinate Branch
        self.conv_shared = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.GELU()
        )
        self.conv_h = nn.Conv2d(mid_channels, in_channels, 1)
        self.conv_w = nn.Conv2d(mid_channels, in_channels, 1)
        self.sigmoid = nn.Sigmoid()
        self._init_weights()

    def _init_weights(self):
        # Zero Init 策略：保证初始状态不破坏 ViT 权重
        for m in [self.conv_h, self.conv_w]:
            nn.init.constant_(m.weight, 0)
            nn.init.constant_(m.bias, -5.0)
        nn.init.constant_(self.mlp[-2].weight, 0)
        nn.init.constant_(self.mlp[-2].bias, -5.0)

    def forward(self, x):
        identity = x
        b, c, h, w = x.size()
        
        # Spectral
        low = self.avg_pool(x)
        high = x - low
        w_spec = self.mlp(self.gap(high).view(b, c)).view(b, c, 1, 1)
        
        # Coordinate
        x_h = F.adaptive_avg_pool2d(x, (h, 1))
        x_w = F.adaptive_avg_pool2d(x, (1, w))
        cat = torch.cat([x_h, x_w.permute(0, 1, 3, 2)], dim=2)
        f = self.conv_shared(cat)
        f_h, f_w = torch.split(f, [h, w], dim=2)
        a_h = self.sigmoid(self.conv_h(f_h))
        a_w = self.sigmoid(self.conv_w(f_w.permute(0, 1, 3, 2)))
        
        # Residual Fusion
        out = identity + identity * (w_spec * a_h * a_w)
        return out

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

# ==========================================
# 3. 模型定义 (Added Projection Head)
# ==========================================
class ViT_AFRN_Model(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        self.embed_dim = 768
        
        self.afrn = SpectralCoordinateAttention(in_channels=self.embed_dim)
        self.gem = GeMPooling(p=3)
        self.head_norm = nn.LayerNorm(self.embed_dim)
        
        # [New] Projection Head for Contrastive Learning
        # 把 768 维降到 128 维，用于计算 Contrastive Loss (SimCLR 论文建议)
        self.projector = nn.Sequential(
            nn.Linear(self.embed_dim, self.embed_dim),
            nn.GELU(),
            nn.Linear(self.embed_dim, 128) # Contrastive embedding dim
        )
        
        self.head = nn.Linear(self.embed_dim, num_classes)

    def forward(self, x, return_feat=False):
        x = self.backbone.forward_features(x)
        if x.shape[1] == 197: x = x[:, 1:, :]
        
        b, n, c = x.shape
        h = w = int(n**0.5)
        x = x.permute(0, 2, 1).view(b, c, h, w)
        
        x = self.afrn(x)
        x = self.gem(x).flatten(1)
        x = self.head_norm(x)
        
        logits = self.head(x)
        
        if return_feat:
            # 返回归一化的特征用于 Contrastive Loss
            proj_feat = F.normalize(self.projector(x), dim=1)
            return logits, proj_feat
        
        return logits

# ==========================================
# 4. 数据加载 (Same as before)
# ==========================================
def get_dataloaders():
    # 稍微增强一下 Augmentation，配合 Contrastive Loss 效果更好
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3), # 加强颜色抖动
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        transforms.RandomErasing(p=0.15)
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    try:
        train_path = os.path.join(DATA_DIR, 'train')
        if not os.path.exists(train_path): train_path = os.path.join(DATA_DIR, 'original/train')
        
        train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
        val_path = train_path.replace('train', 'test')
        val_dataset = datasets.ImageFolder(val_path, transform=val_transform)
    except:
        return None, None, None

    targets = train_dataset.targets
    cw = compute_class_weight('balanced', classes=np.unique(targets), y=targets)
    class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True) # drop_last=True 防止 batch只有1个样本报错
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, class_weights

# ==========================================
# 5. 训练循环 (Updated Loss Logic)
# ==========================================
def main():
    train_loader, val_loader, class_weights = get_dataloaders()
    if train_loader is None: return

    model = ViT_AFRN_Model(num_classes=NUM_CLASSES).to(DEVICE)
    
    # 优化器
    param_groups = [
        {'params': model.backbone.parameters(), 'lr': 1e-5},
        {'params': model.afrn.parameters(), 'lr': 1e-4},
        {'params': model.projector.parameters(), 'lr': 1e-4}, # 新加的 Projector
        {'params': model.head.parameters(), 'lr': 1e-4},
    ]
    optimizer = optim.AdamW(param_groups, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    
    # 定义两个 Loss
    criterion_ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    criterion_supcon = SupConLoss(temperature=TEMP)
    
    best_acc = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for imgs, labels in loop:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            
            # Forward: 获取 logits 和 features
            logits, features = model(imgs, return_feat=True)
            
            # Loss 1: 分类损失 (Cross Entropy)
            loss_ce = criterion_ce(logits, labels)
            
            # Loss 2: 对比损失 (Supervised Contrastive)
            loss_con = criterion_supcon(features, labels)
            
            # Combined Loss
            loss = loss_ce + LAMBDA_CONT * loss_con
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            _, preds = torch.max(logits, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
            loop.set_postfix(
                loss=f"{loss.item():.4f}", 
                ce=f"{loss_ce.item():.3f}", 
                con=f"{loss_con.item():.3f}" # 观察一下对比损失是不是在降
            )
            
        scheduler.step()
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                logits = model(imgs) # Val 不需要 features
                _, preds = torch.max(logits, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        
        val_acc = val_correct / val_total
        print(f"📊 Epoch {epoch+1}: Train Acc: {train_correct/train_total:.4f}, Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "ss_vlm_supcon_best.pth")
            print(f"🎉 New Best Accuracy: {best_acc:.4f} (Saved)")

if __name__ == '__main__':
    main()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

🔥 Running SS-VLM with Supervised Contrastive Loss on cuda...


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Epoch 1/40: 100%|██████████| 383/383 [03:57<00:00,  1.61it/s, ce=1.527, con=3.213, loss=1.8480]


📊 Epoch 1: Train Acc: 0.6161, Val Acc: 0.7223
🎉 New Best Accuracy: 0.7223 (Saved)


...

📊 Epoch 38: Train Acc: 0.9807, Val Acc: 0.8475


Epoch 39/40: 100%|██████████| 383/383 [03:58<00:00,  1.61it/s, ce=1.003, con=2.423, loss=1.2455]


📊 Epoch 39: Train Acc: 0.9790, Val Acc: 0.8579


Epoch 40/40: 100%|██████████| 383/383 [03:58<00:00,  1.61it/s, ce=0.754, con=1.750, loss=0.9293]


📊 Epoch 40: Train Acc: 0.9838, Val Acc: 0.8367
